In [ ]:
import numpy as np
import nibabel as nib
import tensorflow as tf
from tensorflow.keras import layers , models
from sklearn.model_selection import train_test_split
import os
import datetime


In [ ]:
# loader file

def load_nifti(path):
    nii = nib.load(path)
    return nii.get_fdata().astype(np.float32)


# ---------- PREPROCESS ----------

def preprocess(volume, mask):
    # (H, W, D) → (D, H, W)
    volume = np.transpose(volume, (2, 0, 1))
    mask   = np.transpose(mask, (2, 0, 1))

    # normalize volume
    volume = (volume - np.mean(volume)) / (np.std(volume) + 1e-8)

    # binarize mask
    mask = (mask > 0).astype(np.float32)

    return volume, mask



def load_volume(volume_path, mask_path):
    volume = load_nifti(volume_path)
    mask   = load_nifti(mask_path)

    volume, mask = preprocess(volume, mask)

    return volume, mask

In [ ]:
# sampler file

def extract_patch(img, msk, patch_size=48, min_vessel_voxels=50,bg_threshold=20, vessel_prob=0.7, max_tries=10):

    D, H, W = img.shape

    if isinstance(patch_size, int):
        pd = ph = pw = patch_size
    else:
        pd, ph, pw = patch_size

    msk = (msk > 0).astype(np.uint8)
    vessel_coords = np.argwhere(msk > 0)

    if len(vessel_coords) > 0 and np.random.rand() < vessel_prob:
        # vessel-centered
        for _ in range(max_tries):
            z, y, x = vessel_coords[np.random.randint(len(vessel_coords))]

            d = np.clip(z - pd//2, 0, max(0, D - pd))
            h = np.clip(y - ph//2, 0, max(0, H - ph))
            w = np.clip(x - pw//2, 0, max(0, W - pw))

            ip = img[d:d+pd, h:h+ph, w:w+pw]
            mp = msk[d:d+pd, h:h+ph, w:w+pw]
            
            if np.sum(mp) >= min_vessel_voxels:
                return ip, mp

    else:
        # random
        for _ in range(max_tries):
            d = np.random.randint(0, max(1, D - pd + 1))
            h = np.random.randint(0, max(1, H - ph + 1))
            w = np.random.randint(0, max(1, W - pw + 1))

            ip = img[d:d+pd, h:h+ph, w:w+pw]
            mp = msk[d:d+pd, h:h+ph, w:w+pw]

            if np.sum(mp) <= bg_threshold:
                return ip, mp

    return ip, mp


In [ ]:
#generator file 

import gc

def data_generator(image_paths, mask_paths, patch_size=96, min_vessel_voxels=200, batch_size=1):
    n = len(image_paths)

    while True:
        imgs = []
        msks = []

        for _ in range(batch_size):
            j = np.random.randint(0, n)

            img, msk = load_volume(image_paths[j], mask_paths[j])

            ip, mp = extract_patch(
                img, msk,
                patch_size=patch_size,
                min_vessel_voxels=min_vessel_voxels
            )

            # ---- FIXES ----

            ip = ip.astype(np.float32)
            mp = mp.astype(np.float32)

            # normalize mask if needed
            if mp.max() > 1:
                mp = mp / 255.0

            # FORCE channel dim (no conditions)
            ip = ip[..., np.newaxis]
            mp = mp[..., np.newaxis]

            # strict shape check
            assert ip.shape == mp.shape, f"{ip.shape} vs {mp.shape}"

            imgs.append(ip)
            msks.append(mp)

        imgs = np.stack(imgs, axis=0)
        msks = np.stack(msks, axis=0)

        yield imgs, msks
        del imgs, msks, ip, mp, img, msk
        gc.collect()


In [ ]:
#3D-unet model file


def conv_block(x, f):
    shortcut = x

    x = layers.Conv3D(f, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Conv3D(f, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)

    if shortcut.shape[-1] != f:
        shortcut = layers.Conv3D(f, 1, padding='same')(shortcut)

    x = layers.Add()([x, shortcut])
    x = layers.ReLU()(x)
    return x


def build_unet_3d(input_shape=(32,32,32,1), base_filters=16):
    f = base_filters

    inputs = layers.Input(input_shape)

    # Encoder
    c1 = conv_block(inputs, f)
    p1 = layers.MaxPooling3D()(c1)

    c2 = conv_block(p1, f*2)
    p2 = layers.MaxPooling3D()(c2)

    c3 = conv_block(p2, f*4)
    p3 = layers.MaxPooling3D()(c3)

    # Bottleneck
    bn = conv_block(p3, f*8)

    # Decoder
    u3 = layers.UpSampling3D()(bn)
    u3 = layers.Concatenate()([u3, c3])
    c4 = conv_block(u3, f*4)

    u2 = layers.UpSampling3D()(c4)
    u2 = layers.Concatenate()([u2, c2])
    c5 = conv_block(u2, f*2)

    u1 = layers.UpSampling3D()(c5)
    u1 = layers.Concatenate()([u1, c1])
    c6 = conv_block(u1, f)

    outputs = layers.Conv3D(1, 1, activation='sigmoid')(c6)

    return tf.keras.Model(inputs, outputs)

In [ ]:
# loss function file

def dice_loss(y_true, y_pred, smooth=1e-6):
    y_true = tf.reshape(y_true, [-1])
    y_pred = tf.reshape(y_pred, [-1])

    intersection = tf.reduce_sum(y_true * y_pred)
    return 1 - (2. * intersection + smooth) / (
        tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) + smooth
    )

def focal_loss(y_true, y_pred, gamma=2.0, alpha=0.25):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    # this already keeps channel dimension correctly
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)

    # ensure SAME rank as y_true (not blindly expanding)
    if len(bce.shape) < len(y_true.shape):
        bce = tf.expand_dims(bce, axis=-1)

    p_t = y_true * y_pred + (1 - y_true) * (1 - y_pred)

    return alpha * tf.pow(1 - p_t, gamma) * bce


def combined_loss(y_true, y_pred):
    return dice_loss(y_true, y_pred) + tf.reduce_mean(focal_loss(y_true, y_pred))

In [ ]:
#GPU MEMORY GROWTH
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

gpus = tf.config.list_physical_devices('GPU')

if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

    print("GPU detected:", gpus)
else:
    print("No GPU detected, running on CPU")

In [ ]:
patch_size = 96
min_vessel_voxels = 200

images_dir = "/home/vatsal/projects/3D-unet-costa/costa/COSTA_dataset_v1/COSTA-Dataset-v1/imagesTr/ADAM"
labels_dir = "/home/vatsal/projects/3D-unet-costa/costa/COSTA_dataset_v1/COSTA-Dataset-v1/labelsTr/ADAM"

image_files = sorted([f for f in os.listdir(images_dir) if f.endswith(".mhd")])

image_paths = [os.path.join(images_dir, f) for f in image_files]
mask_paths  = [os.path.join(labels_dir, f) for f in image_files]


In [ ]:
# TEST TRAIN SPLIT 

train_img, val_img, train_mask, val_mask = train_test_split(
    image_paths,
    mask_paths,
    test_size=0.2,
    random_state=42
)

print("Train scans:", len(train_img))
print("Val scans:", len(val_img))

In [ ]:
# DATA GENERATORS

train_gen = data_generator(
    train_img,
    train_mask,
    patch_size=patch_size,
    min_vessel_voxels=min_vessel_voxels,

)

val_gen = data_generator(
    val_img,
    val_mask,
    patch_size=patch_size,
    min_vessel_voxels=min_vessel_voxels
)


In [ ]:
# TF DATASETS

train_dataset = tf.data.Dataset.from_generator(
    lambda: train_gen,
    output_signature=(
        tf.TensorSpec(shape=(None,patch_size,patch_size,patch_size,1), dtype=tf.float32),
        tf.TensorSpec(shape=(None,patch_size,patch_size,patch_size,1), dtype=tf.float32)
    )
)

val_dataset = tf.data.Dataset.from_generator(
    lambda: val_gen,
    output_signature=(
        tf.TensorSpec(shape=(None,patch_size,patch_size,patch_size,1), dtype=tf.float32),
        tf.TensorSpec(shape=(None,patch_size,patch_size,patch_size,1), dtype=tf.float32)
    )
)


In [ ]:
# SAVED MODELS FOLDER CREATION
def create_model_dir(base_path="/home/vatsal/projects/3D-unet-costa/models"):

    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")

    model_dir = os.path.join(base_path, f"unet3d_{timestamp}")

    os.makedirs(model_dir, exist_ok=True)

    return model_dir


model_dir = create_model_dir()

In [ ]:
#CALLBACKS

callbacks = [

    tf.keras.callbacks.ModelCheckpoint(
        filepath=os.path.join(model_dir,"best_model.keras"),
        monitor="val_loss",
        save_best_only=True,
        mode="min",
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    ),

    tf.keras.callbacks.TensorBoard(
        log_dir=os.path.join(model_dir,"logs"),
        histogram_freq=1
    )
]



In [ ]:
model = build_unet_3d(
    input_shape=(patch_size,patch_size,patch_size,1),
    base_filters=32
)


model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss=combined_loss,
    metrics=[
        dice_loss,
        tf.keras.metrics.Precision(thresholds=0.5),
        tf.keras.metrics.Recall(thresholds=0.5)
    ]
)


model.summary()

In [ ]:
print("Starting training...")


history = model.fit(

    train_dataset,

    validation_data=val_dataset,

    steps_per_epoch=100,

    validation_steps=30,

    epochs=100,

    callbacks=callbacks
)


In [ ]:
# SAVING FINAL MODEL
model.save(os.path.join(model_dir,"final_model.keras"))

print("Training complete")